In [ ]:
pip install torch torchvision numpy pillow scikit-image

# Adversarial Attack on ResNet-18

An untargeted PGD attack on an ImageNet-pretrained ResNet-18 classifier: a small, L-inf-bounded pixel perturbation is enough to flip the model's predicted class while the image still looks unchanged to a human.

## 1. Setup: model, helper functions, and the attack

Two well-known test images bundled with `scikit-image` (`chelsea` - a cat, `coffee` - a cup of espresso) stand in for arbitrary input photos, so the notebook needs no external image downloads or licensing questions.

In [ ]:
import json
import os

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from skimage import data as skdata
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torchvision.models import ResNet18_Weights, resnet18

WEIGHTS = ResNet18_Weights.IMAGENET1K_V1
CATEGORIES = WEIGHTS.meta["categories"]
PREPROCESS = WEIGHTS.transforms()
MEAN = torch.tensor(PREPROCESS.mean).view(3, 1, 1)
STD = torch.tensor(PREPROCESS.std).view(3, 1, 1)


def load_sample(name):
    arr = getattr(skdata, name)()
    return Image.fromarray(arr).convert("RGB")


def to_model_input(img_0_1):
    """img_0_1: float tensor CHW in [0,1] -> resized, ImageNet-normalized model input."""
    resized = F.interpolate(img_0_1.unsqueeze(0), size=(224, 224), mode="bilinear", align_corners=False).squeeze(0)
    return ((resized - MEAN) / STD).unsqueeze(0)


def predict(model, img_0_1):
    with torch.no_grad():
        logits = model(to_model_input(img_0_1))
        probs = F.softmax(logits, dim=-1)[0]
    top_prob, top_idx = torch.max(probs, dim=0)
    return CATEGORIES[top_idx.item()], top_prob.item(), top_idx.item()

In [ ]:
def run_pgd_attack(model, img_0_1, epsilon, alpha, num_iter):
    """Untargeted PGD in pixel space [0,1], L-inf bounded by epsilon.

    Unlike a targeted attack (push toward a chosen wrong class), this just maximizes the loss on the
    model's own original prediction, so it finds whatever nearby class the decision boundary is
    weakest against.
    """
    orig = img_0_1.clone().detach()
    _, _, true_idx = predict(model, orig)
    target = torch.tensor([true_idx])

    adv = orig.clone().detach()
    for _ in range(num_iter):
        adv.requires_grad_(True)
        logits = model(to_model_input(adv))
        loss = F.cross_entropy(logits, target)
        grad = torch.autograd.grad(loss, adv)[0]

        with torch.no_grad():
            adv = adv + alpha * grad.sign()
            perturbation = torch.clamp(adv - orig, min=-epsilon, max=epsilon)
            adv = torch.clamp(orig + perturbation, min=0, max=1)

    return adv.detach()

## 2. Load the classifier

In [ ]:
model = resnet18(weights=WEIGHTS)
model.eval()

## 3. Run the attack across samples and perturbation budgets

Two `epsilon` budgets, both well below what a human notices at a glance: `mild` (8/255) and `strong` (16/255). Raw and adversarial images are saved under `samples/resnet18/`.

In [ ]:
SAMPLES_DIR = "samples/resnet18"
os.makedirs(SAMPLES_DIR, exist_ok=True)

SAMPLE_STEMS = ["chelsea", "coffee"]
CONFIGS = [
    {"name": "mild", "epsilon": 8 / 255, "alpha": 2 / 255, "num_iter": 10},
    {"name": "strong", "epsilon": 16 / 255, "alpha": 4 / 255, "num_iter": 10},
]

results = []
for stem in SAMPLE_STEMS:
    img = load_sample(stem)
    img_0_1 = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
    orig_label, orig_conf, _ = predict(model, img_0_1)
    img.save(f"{SAMPLES_DIR}/{stem}_raw.png")

    for cfg in CONFIGS:
        print(f"running {stem} / {cfg['name']} ...")
        adv_0_1 = run_pgd_attack(model, img_0_1, cfg["epsilon"], cfg["alpha"], cfg["num_iter"])
        adv_label, adv_conf, _ = predict(model, adv_0_1)

        diff = (adv_0_1 - img_0_1).numpy()
        orig_np, adv_np = img_0_1.numpy(), adv_0_1.numpy()
        psnr = peak_signal_noise_ratio(orig_np, adv_np, data_range=1.0)
        ssim = structural_similarity(
            orig_np.transpose(1, 2, 0), adv_np.transpose(1, 2, 0), channel_axis=2, data_range=1.0
        )

        metrics = {
            "sample": stem,
            "config": cfg["name"],
            "epsilon_255": round(cfg["epsilon"] * 255),
            "original_label": orig_label,
            "original_confidence": orig_conf,
            "adversarial_label": adv_label,
            "adversarial_confidence": adv_conf,
            "misclassified": orig_label != adv_label,
            "linf_perturbation_255": float(np.max(np.abs(diff)) * 255),
            "l2_perturbation": float(np.linalg.norm(diff)),
            "psnr_db": float(psnr),
            "ssim": float(ssim),
        }
        results.append(metrics)

        adv_uint8 = (adv_np.transpose(1, 2, 0) * 255).astype(np.uint8)
        Image.fromarray(adv_uint8).save(f"{SAMPLES_DIR}/{stem}_{cfg['name']}_adversarial.png")

        print(f"[{stem} / {cfg['name']}] epsilon={metrics['epsilon_255']}/255")
        print(f"  original:    {orig_label} ({orig_conf:.3f})")
        print(f"  adversarial: {adv_label} ({adv_conf:.3f})")
        print(f"  misclassified={metrics['misclassified']}  Linf={metrics['linf_perturbation_255']:.1f}/255"
              f"  PSNR={psnr:.1f}dB  SSIM={ssim:.4f}")
        print()

with open(f"{SAMPLES_DIR}/results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("wrote results.json")

running chelsea / mild ...
[chelsea / mild] epsilon=8/255
  original:    Egyptian cat (0.769)
  adversarial: Persian cat (1.000)
  misclassified=True  Linf=8.0/255  PSNR=34.0dB  SSIM=0.8893

running chelsea / strong ...
[chelsea / strong] epsilon=16/255
  original:    Egyptian cat (0.769)
  adversarial: Persian cat (1.000)
  misclassified=True  Linf=16.0/255  PSNR=28.2dB  SSIM=0.7029

running coffee / mild ...
[coffee / mild] epsilon=8/255
  original:    espresso (0.990)
  adversarial: Irish setter (1.000)
  misclassified=True  Linf=8.0/255  PSNR=35.1dB  SSIM=0.8933

running coffee / strong ...
[coffee / strong] epsilon=16/255
  original:    espresso (0.990)
  adversarial: Irish setter (1.000)
  misclassified=True  Linf=16.0/255  PSNR=29.5dB  SSIM=0.7266

wrote results.json


## 4. Results

All four configurations flip the top-1 prediction, and both `strong` runs push the model to 100% confidence in the *wrong* class while `PSNR`/`SSIM` stay in a range generally considered high-fidelity (30+dB, 0.7+ SSIM) - the perturbation is a targeted, structured signal, not random noise, so it moves the model far more than it moves a human's perception of the image.

Raw and adversarial `.png` files are committed under `samples/resnet18/` - open them on GitHub to compare directly.

In [ ]:
print(f"{'sample':<10}{'config':<8}{'eps/255':<9}{'orig -> adv':<45}{'PSNR':<8}{'SSIM':<8}")
for r in results:
    orig_adv = f"{r['original_label']} -> {r['adversarial_label']}"
    print(f"{r['sample']:<10}{r['config']:<8}{r['epsilon_255']:<9}{orig_adv:<45}{r['psnr_db']:<8.1f}{r['ssim']:<8.4f}")

sample    config  eps/255  orig -> adv                                  PSNR    SSIM    
chelsea   mild    8        Egyptian cat -> Persian cat                  34.0    0.8893  
chelsea   strong  16       Egyptian cat -> Persian cat                  28.2    0.7029  
coffee    mild    8        espresso -> Irish setter                     35.1    0.8933  
coffee    strong  16       espresso -> Irish setter                     29.5    0.7266  
